In [9]:
from ukrdc_stats.utils import ukrdc_connection_from_env, cache_connection_from_env
from ukrdc_stats.calculators.chronic_kidney_disease import ChronicKidneyDiseaseBase
import datetime as dt 

facility = "RH8"
ukrdc3 = ukrdc_connection_from_env()
ukrdc_stats_cache = cache_connection_from_env()

# run function to extract stats
ukrdc_stats_cache.flushall()

ckd_calculator = ChronicKidneyDiseaseBase(ukrdc3, ukrdc_stats_cache, facility, dt.datetime(2023,1,1))

ckd_calculator.extract_stats()
#print(ckd_calculator._patient_cohort.head())
ckd_calculator.renal_diagnosis([facility])

(0.025350837483023993, 2209)


,diagnosiscode,destination_code,diagnosiscodestd,creation_date
0,1752,Glomerulonephritis,EDTA2,2024-02-15 14:11:18.372892
7,1752,Other,EDTA2,2024-02-15 14:11:18.372892
38,1752,Renal vascular disease,EDTA2,2024-02-15 14:11:18.372892
60,1752,Pyelonephritis,EDTA2,2024-02-15 14:11:18.372892
63,1752,Polycystic kidney disease,EDTA2,2024-02-15 14:11:18.372892
...,...,...,...,...
8250,3555,Pyelonephritis,EDTA2,2024-02-15 16:23:54.745554
8253,3555,Polycystic kidney disease,EDTA2,2024-02-15 16:23:54.745554
8321,3555,Diabetes,EDTA2,2024-02-15 16:23:54.745554
8325,3555,Hypertension,EDTA2,2024-02-15 16:23:54.745554


In [10]:
from ukrdc_stats.calculators.chronic_kidney_disease import ChronicKidneyDiseaseScaleCompare

ckd_multiscale = ChronicKidneyDiseaseScaleCompare(ukrdc3, ukrdc_stats_cache, "RNJ00",["RJZ","RAJ","RNJ00"], dt.datetime(2023,1,1))

ckd_multiscale.extract_stats()
funnel_plot = ckd_multiscale.assemble_funnel()
min_pop = min(funnel_plot['populations'])
n_points = 3
pop_line = [min_pop + (funnel_plot['ensemble population'] - min_pop) * i / (n_points-1) for i in range(n_points)]

# do we want dashboard stats to do this? 
limit_plus = [funnel_plot['ensemble population'] + funnel_plot['0.95 fit'] * pop **-.5 for pop in pop_line]
limit_minus = [funnel_plot['ensemble population'] - funnel_plot['0.95 fit'] * pop **-.5 for pop in pop_line]
limit_plus_strict = [funnel_plot['ensemble population'] + funnel_plot['0.99 fit'] * pop **-.5 for pop in pop_line]
limit_minus_strict = [funnel_plot['ensemble population'] - funnel_plot['0.99 fit'] * pop **-.5 for pop in pop_line]


print(pop_line)


[463.0, 1954.0, 3445.0]


In [14]:
import plotly.graph_objects as go
from ukrdc_stats.calculators.chronic_kidney_disease import ChronicKidneyDiseaseScaleCompare



renal_units = ["RJZ", "RAJ", "RNJ00", "RJE01", "RK7CC"]


# Assuming you already have the necessary data and objects defined
ckd_multiscale = ChronicKidneyDiseaseScaleCompare(ukrdc3, ukrdc_stats_cache, "RNJ00", renal_units,
                                                 dt.datetime(2024, 6, 1))



ckd_multiscale.extract_stats()
funnel_data = ckd_multiscale.assemble_funnel()

# Calculate pop_line and limits
min_pop = min(funnel_data['populations'])
n_points = 50
#pop_line = [min_pop + (funnel_data['ensemble proportion'] - min_pop) * i / (n_points - 1) for i in range(n_points)]
pop_line = [i for i in range(100, 2000)]


limit_plus = [funnel_data['ensemble proportion'] + funnel_data['0.95 fit'] * pop ** -0.5 for pop in pop_line]
limit_minus = [funnel_data['ensemble proportion'] - funnel_data['0.95 fit'] * pop ** -0.5 for pop in pop_line]
limit_plus_strict = [funnel_data['ensemble proportion'] + funnel_data['0.99 fit'] * pop ** -0.5 for pop in pop_line]
limit_minus_strict = [funnel_data['ensemble proportion'] - funnel_data['0.99 fit'] * pop ** -0.5 for pop in pop_line]

# Create the plot
fig = go.Figure()

# Add the lines to the plot
fig.add_trace(go.Scatter(x=pop_line, y=limit_plus, name='0.95 Fit +', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=pop_line, y=limit_minus, name='0.95 Fit -', line=dict(color='red')))
fig.add_trace(go.Scatter(x=pop_line, y=limit_plus_strict, name='0.99 Fit +', line=dict(color='green')))
fig.add_trace(go.Scatter(x=pop_line, y=limit_minus_strict, name='0.99 Fit -', line=dict(color='purple')))


# Add individual data points as markers
fig.add_trace(go.Scatter(x=funnel_data['populations'], y=funnel_data['proportions'], mode='markers', name='Individual Points',
                         marker=dict(size=10, color='black', symbol='circle'),
                         text=funnel_data['facilities'], # Facility names as labels for each point
                         textposition='bottom center', # Position of the labels relative to the points
                         hoverinfo='text' # Show the facility names on hover
                        ))
# Update the layout of the plot
fig.update_layout(
    title="Funnel Plot",
    xaxis_title="Population",
    yaxis_title="Ensemble Population",
    #legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

# Show the plot
fig.show()

In [12]:
from ukrdc_stats.calculators.chronic_kidney_disease import ChronicKidneyDiseaseLongditudinal

ckd_long = ChronicKidneyDiseaseLongditudinal(ukrdc3, ukrdc_stats_cache,"RJZ", 12, dt.datetime(2023,1,1))
ckd_long.extract_stats()

(0.6666666666666666, 435)
(0.6681818181818182, 440)
(0.6621621621621622, 444)
(0.6614699331848553, 449)
(0.6533333333333333, 450)
(0.6518847006651884, 451)
(0.668141592920354, 452)
(0.6622516556291391, 453)
(0.6615384615384615, 455)
(0.6651982378854625, 454)
(0.6710526315789473, 456)
(0.665938864628821, 458)


In [13]:
from ukrdc_stats.utils import subtract_months



print(subtract_months(dt.datetime(2023,4,1), 24))

2021-04-01 00:00:00
